In [1]:
from kafka import KafkaProducer
import pandas as pd
from models import json_serializer, ride_from_row
import dataclasses
import time

In [2]:
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet"
raw_df = pd.read_parquet(url)

In [14]:
raw_df.shape

(49416, 21)

In [3]:
server = "localhost:9092"
producer = KafkaProducer(
    bootstrap_servers=[server], 
    value_serializer=json_serializer
)

In [24]:
# There are numpy types in the ride dataclass that are not JSON serializable,
# so its necessary to convert them to native python types, which is what the numpy_encoder function does.
topic_name = 'rides'
first_ride = ride_from_row(raw_df.iloc[1])
producer.send(topic_name, dataclasses.asdict(first_ride))
producer.flush()

In [ ]:
# topic_name = 'green_taxi_rides'
t0 = time.time()
for _, row in raw_df.iterrows():
    if row.passenger_count.isna():
        continue
    ride = ride_from_row(row)
    producer.send('green-trips', dataclasses.asdict(ride))
producer.flush()
t1 = time.time()
print(f"Time taken: {t1 - t0} seconds")

Time taken: 19.878994703292847 seconds


In [6]:
raw_df.shape

(49416, 21)